In [1]:
from torch.utils.data import DataLoader, random_split 
from firealarm_net import FireAlarmCRNN, MelDataset
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder 
import torch.nn as nn
import numpy as np 
import torch 
import os

In [2]:
# Set seed to have the same outputs constantly when testing
seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

In [3]:
# Load Data by traversing through the features
features = []
labels = []

print("Loading data...")
for filename in sorted(os.listdir("features")):
    if filename.endswith(".npy") and not filename.endswith("_rf.npy"):
        # Load the data
        file_path = os.path.join("features", filename)
        data = np.load(file_path)

        # Simple label extraction based on filename
        if "fire" in filename: 
            label = "fire_alarm"
            count = 1  
        elif "siren" in filename: 
            label = "siren"
            count = 1
        elif "appliance" in filename: 
            label = "appliance"
            count = 1
        elif "carbon" in filename:
            label = "carbon_alarm"
            count = 15 # Duplicate carbon samples 15 times 
        else:
            continue
        
        for _ in range(count):
            features.append(data)
            labels.append(label)

# Convert to Tensors
features = np.array(features)

# Standardize data (make mean 0, std 1) for better learning
features = (features - features.mean()) / (features.std() + 1e-8)

# Add channel dim: (N, 64, 157) -> (N, 1, 64, 157)
features = features[:, None, :, :]

# Encode Labels (appliance -> 0, carbon_alarm -> 1, fire_alarm -> 2, siren -> 3)
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

print(f"Data loaded: {len(features)} samples.")

Loading data...
Data loaded: 46855 samples.


In [4]:
# Creating model directory
os.makedirs("model", exist_ok=True) 

# Preparing for our Model
# Dataset & Loader
dataset = MelDataset(features, labels_encoded)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size], generator=torch.Generator().manual_seed(seed))

# Create a generator for DataLoader shuffling
g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# Model setup
model = FireAlarmCRNN(num_classes=len(encoder.classes_))
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Order of classes: [Appliance, Carbon, Fire, Siren]
# We make the model care 2x to 10x more about Fire and Carbon than the others.
weights = torch.tensor([1.0, 10.0, 2.0, 1.0])
criterion = nn.CrossEntropyLoss(weight=weights)

/opt/anaconda3/envs/Random/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Training the Model
epochs = 15
best_acc = 0.0

print("\nStarting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        # Track training stats
        running_loss += loss.item() * x.size(0)
        correct += (output.argmax(1) == y).sum().item()
        total += y.size(0)
    
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation step
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += y.size(0)
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1}/{epochs}  "
          f"TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "state_dict": model.state_dict(),
            "classes": list(encoder.classes_)
            }, "model/model.pt")    
        

print(f"Done! Best Accuracy: {best_acc:.4f}")


Starting training...
Epoch 1/15  TrainLoss=0.5347  ValLoss=0.3219  TrainAcc=0.6948  ValAcc=0.7921
Epoch 2/15  TrainLoss=0.3235  ValLoss=0.2475  TrainAcc=0.7980  ValAcc=0.8444
Epoch 3/15  TrainLoss=0.2449  ValLoss=0.1802  TrainAcc=0.8410  ValAcc=0.8761
Epoch 4/15  TrainLoss=0.1968  ValLoss=0.1457  TrainAcc=0.8697  ValAcc=0.8974
Epoch 5/15  TrainLoss=0.1445  ValLoss=0.1398  TrainAcc=0.9046  ValAcc=0.9117
Epoch 6/15  TrainLoss=0.1192  ValLoss=0.0929  TrainAcc=0.9235  ValAcc=0.9410
Epoch 7/15  TrainLoss=0.0882  ValLoss=0.0807  TrainAcc=0.9428  ValAcc=0.9470
Epoch 8/15  TrainLoss=0.0769  ValLoss=0.0705  TrainAcc=0.9493  ValAcc=0.9546
Epoch 9/15  TrainLoss=0.0677  ValLoss=0.0562  TrainAcc=0.9559  ValAcc=0.9568
Epoch 10/15  TrainLoss=0.0585  ValLoss=0.0587  TrainAcc=0.9624  ValAcc=0.9639
Epoch 11/15  TrainLoss=0.0489  ValLoss=0.0610  TrainAcc=0.9677  ValAcc=0.9649
Epoch 12/15  TrainLoss=0.0427  ValLoss=0.0594  TrainAcc=0.9731  ValAcc=0.9640
Epoch 13/15  TrainLoss=0.0399  ValLoss=0.0527  Trai

In [6]:
# Load the model to evaluate
checkpoint = torch.load("model/model.pt")
model.load_state_dict(checkpoint["state_dict"])
model.eval()

all_preds = []
all_labels = []

print("Evaluating best CNN model on validation set...")

# Run the validation data through the model
with torch.no_grad():
    for x, y in val_loader:
        outputs = model(x)
        preds = outputs.argmax(dim=1) # Get the index of the highest probability
        
        # Store predictions and true labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Print the classification report
# We can use encoder.classes_ to automatically get the category names
print("\nCNN Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=encoder.classes_))

Evaluating best CNN model on validation set...

CNN Classification Report:

              precision    recall  f1-score   support

   appliance       0.98      0.96      0.97      3316
carbon_alarm       1.00      1.00      1.00      1093
  fire_alarm       0.93      0.99      0.96      1629
       siren       0.98      0.98      0.98      3333

    accuracy                           0.98      9371
   macro avg       0.97      0.98      0.98      9371
weighted avg       0.98      0.98      0.98      9371

